In [1]:
import pandas as pd

import numpy as np

read in .csv

In [2]:
df_2021 = pd.read_csv('DC-migration-2021.csv')


I want to find some overall migration numbers for the narrative part of my story.

First, in 2021 as a sample year, total migration to DC:

Can see from original unparced data that total migration to DC is 57,746 +/- 5,874 in 2021.


Now I want to see how that figure compares to other states, adjusted per capita.

I want to take that original excel file and pull out the total migration from each state and compare to population.

Repeating parsing from the DC-migration-2021-ran.ipynb notebook

In [3]:
df = pd.read_excel('ORIGINAL DATA State_to_State_Migrations_Table_2021.xls')

df = df.iloc[np.r_[4:43, 44:77]] #drop top and bottom outside of actual chart in the excel file

df = df.dropna(how="all") #drop all rows with all na values


Rename columns

In [4]:

#for the first column

df.rename(columns={'Table with row headers in column A, L, W, AH, AS, BD, BO, BZ, CK, CV, and DG, and column headers in rows 6 through 8 and 45 through 47.': '0'}, inplace=True)

#for the rest of the columns:

for col in df.columns[1:130]:  # 2–129 (0-based index, so skip first col)
    if str(col).startswith("Unnamed:"):
        df.rename(columns={col: str(col).replace("Unnamed: ", "")}, inplace=True)

df.head()

,0,1,2,3,4,5,6,7,8,9,...,120,121,122,123,124,125,126,127,128,129
4,Current residence in,Population 1 year and over,NaN,Same house 1 year ago,NaN,Same state of residence 1 year ago,NaN,Different state of residence 1 year ago,NaN,NaN,...,NaN,Current residence in,Abroad 1 year ago,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Total,NaN,Alabama,...,NaN,NaN,Total,NaN,Puerto Rico,NaN,U.S. Island Area,NaN,Foreign Country,NaN
6,NaN,Estimate,MOE,Estimate,MOE,Estimate,MOE,Estimate,MOE,Estimate,...,MOE,NaN,Estimate,MOE,Estimate,MOE,Estimate,MOE,Estimate,MOE
8,United States2,328464538,34090,286552923,190336,32577121,167671,7859837,75123,104481,...,6515,United States2,1474657,34636,54669,7470,19587,4066,1400401,33711
10,Alabama,4989797,4341,4392571,25089,468964,23356,115641,10300,NaN,...,216,Alabama,12621,2388,534,436,29,47,12058,2400


Drop the whole end of the dataframe, as I wont be needing for this 

In [5]:
df = df.loc[:, :'8'] 

Preserve NA values as 0s 

In [6]:
df.iloc[3:] = df.iloc[3:].fillna('0')

df.head()

,0,1,2,3,4,5,6,7,8
4,Current residence in,Population 1 year and over,NaN,Same house 1 year ago,NaN,Same state of residence 1 year ago,NaN,Different state of residence 1 year ago,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Total,NaN
6,NaN,Estimate,MOE,Estimate,MOE,Estimate,MOE,Estimate,MOE
8,United States2,328464538,34090,286552923,190336,32577121,167671,7859837,75123
10,Alabama,4989797,4341,4392571,25089,468964,23356,115641,10300


Rename columns and drop columns I don't need.
I want to keep only three columns: statename, Population 1 year and over, and Different state of residence 1 year ago -- total.
I can calucate the rest from there.

In [7]:
#first, dealing with columns B-G/1-6)

column_0_name = 'state'

column_1_name = 'population 1 year and over (estimate)'
column_2_name = 'population 1 year and over (moe)'

column_5_name = 'Same state of residence 1 year ago (estimate)'
column_6_name = 'Same state of residence 1 year ago (moe)'

column_7_name = 'Different state of residence 1 year ago (estimate)'
column_8_name = 'Different state of residence 1 year ago (moe)'

df_clean = df.rename(columns ={
    '0': column_0_name,
    '1': column_1_name,
    '2': column_2_name,
    '5': column_5_name,
    '6': column_6_name,
    '7': column_7_name,
    '8': column_8_name})

df = df_clean

df.head()

,state,population 1 year and over (estimate),population 1 year and over (moe),3,4,Same state of residence 1 year ago (estimate),Same state of residence 1 year ago (moe),Different state of residence 1 year ago (estimate),Different state of residence 1 year ago (moe)
4,Current residence in,Population 1 year and over,NaN,Same house 1 year ago,NaN,Same state of residence 1 year ago,NaN,Different state of residence 1 year ago,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Total,NaN
6,NaN,Estimate,MOE,Estimate,MOE,Estimate,MOE,Estimate,MOE
8,United States2,328464538,34090,286552923,190336,32577121,167671,7859837,75123
10,Alabama,4989797,4341,4392571,25089,468964,23356,115641,10300


In [8]:


#drop columns I dont need (same house as 1 year ago estimate and MOE columns)
df.drop(['3', '4'], axis=1, inplace=True)



Delete subtitle rows

In [9]:
df = df.iloc[3:]

df.head()

,state,population 1 year and over (estimate),population 1 year and over (moe),Same state of residence 1 year ago (estimate),Same state of residence 1 year ago (moe),Different state of residence 1 year ago (estimate),Different state of residence 1 year ago (moe)
8,United States2,328464538,34090,32577121,167671,7859837,75123
10,Alabama,4989797,4341,468964,23356,115641,10300
11,Alaska,723949,1517,63611,6501,31378,3888
12,Arizona,7202745,5156,759450,24263,264948,14419
13,Arkansas,2990311,2937,312848,15129,76108,8024


In [10]:
df['Different state of residence 1 year ago (estimate)'] = pd.to_numeric(df['Different state of residence 1 year ago (estimate)'], errors='coerce')

df['population 1 year and over (estimate)'] = pd.to_numeric(df['population 1 year and over (estimate)'], errors='coerce')

df['new_people_adj'] = df['Different state of residence 1 year ago (estimate)'] / df['population 1 year and over (estimate)']

df.head()

,state,population 1 year and over (estimate),population 1 year and over (moe),Same state of residence 1 year ago (estimate),Same state of residence 1 year ago (moe),Different state of residence 1 year ago (estimate),Different state of residence 1 year ago (moe),new_people_adj
8,United States2,328464538.0,34090,32577121,167671,7859837.0,75123,0.023929
10,Alabama,4989797.0,4341,468964,23356,115641.0,10300,0.023175
11,Alaska,723949.0,1517,63611,6501,31378.0,3888,0.043343
12,Arizona,7202745.0,5156,759450,24263,264948.0,14419,0.036784
13,Arkansas,2990311.0,2937,312848,15129,76108.0,8024,0.025452


In [15]:
df.sort_values('new_people_adj', ascending=False)

,state,population 1 year and over (estimate),population 1 year and over (moe),Same state of residence 1 year ago (estimate),Same state of residence 1 year ago (moe),Different state of residence 1 year ago (estimate),Different state of residence 1 year ago (moe),new_people_adj
19,District of Columbia,661026.0,1835,68606,7109,57746.0,5874,0.087358
24,Idaho,1879719.0,2400,176458,11576,96388.0,9806,0.051278
23,Hawaii,1426298.0,2455,110790,9317,71626.0,8474,0.050218
68,Vermont,641007.0,929,42454,4180,31809.0,3854,0.049623
55,North Dakota,764638.0,1475,83572,7158,37844.0,5480,0.049493
74,Wyoming,573476.0,1211,63128,6111,27281.0,4320,0.047571
16,Colorado,5757628.0,4091,686892,20293,250031.0,17084,0.043426
11,Alaska,723949.0,1517,63611,6501,31378.0,3888,0.043343
18,Delaware,994669.0,1764,58180,5692,42551.0,4342,0.042779
48,Nevada,3111722.0,3366,305571,18198,132648.0,11663,0.042628
